# 字符串模糊匹配指南

In [1]:
import difflib
import textdistance

In [36]:
def matches(large_string, query_string, threshold):
    s = difflib.SequenceMatcher(None, large_string, query_string)
    match = ''.join(large_string[i:i+n] for i, j, n in s.get_matching_blocks() if n)
    yield match
    
large_string = "济南春江悦茗"
query_string = "春江悦名"

In [37]:
# # 针对英文
# def matches(large_string, query_string, threshold):
#     words = large_string.split()
#     for word in words:
#         s = difflib.SequenceMatcher(None, word, query_string)
#         match = ''.join(word[i:i+n] for i, j, n in s.get_matching_blocks() if n)
#         if len(match) / float(len(query_string)) >= threshold:
#             yield match

# large_string = "thelargemanhatanproject is a great project in themanhattincity"
# query_string = "manhattan"

In [38]:
for match in matches(large_string, query_string, 0.8):
    if match:
        ratio1 = textdistance.cosine.normalized_similarity(match, query_string)
        ratio1 = round(ratio1, 5)
        if ratio1 >= 0.86:
            print("模糊得分：", ratio1, "\t最长匹配子串：", match, "\t查询词：", query_string)

模糊得分： 0.86603 	最长匹配子串： 春江悦 	查询词： 春江悦名


===

In [15]:
import textdistance

In [16]:
textdistance.levenshtein("this test", "that test") # 2
textdistance.levenshtein.normalized_similarity("this test", "that test") # 0.8

0.7777777777777778

In [17]:
textdistance.cosine("apple", "ppale") # 1.0

1.0

# Fast Fuzzy Matching

In [45]:
import pandas as pd
from tqdm import tqdm
import os
from matplotlib import style
style.use('fivethirtyeight')

In [48]:
names =  pd.read_csv('messy org names.csv',encoding='latin')
print('The shape: %d x %d' % names.shape)
print('There are %d unique values' % names.buyer.shape[0])

The shape: 3651 x 3
There are 3651 unique values


In [ ]:
!pip install ftfy # amazing text cleaning for decode issues..

In [49]:
import re
from ftfy import fix_text

def ngrams(string, n=3):
    string = fix_text(string) # fix text
    string = string.encode("ascii", errors="ignore").decode() #remove non ascii chars
    string = string.lower()
    chars_to_remove = [")","(",".","|","[","]","{","}","'"]
    rx = '[' + re.escape(''.join(chars_to_remove)) + ']'
    string = re.sub(rx, '', string)
    string = string.replace('&', 'and')
    string = string.replace(',', ' ')
    string = string.replace('-', ' ')
    string = string.title() # normalise case - capital at start of each word
    string = re.sub(' +',' ',string).strip() # get rid of multiple spaces and replace with a single
    string = ' '+ string +' ' # pad names for ngrams...
    string = re.sub(r'[,-./]|\sBD',r'', string)
    ngrams = zip(*[string[i:] for i in range(n)])
    return [''.join(ngram) for ngram in ngrams]

print('All 3-grams in "Department":')
print(ngrams('Department'))

All 3-grams in "Department":
[' De', 'Dep', 'epa', 'par', 'art', 'rtm', 'tme', 'men', 'ent', 'nt ']


In [54]:
# pip install sparse_dot_topn

In [51]:
import numpy as np
from scipy.sparse import csr_matrix
!pip install sparse_dot_topn #uncomment to install
import sparse_dot_topn.sparse_dot_topn as ct


def awesome_cossim_top(A, B, ntop, lower_bound=0):
    # force A and B as a CSR matrix.
    # If they have already been CSR, there is no overhead
    A = A.tocsr()
    B = B.tocsr()
    M, _ = A.shape
    _, N = B.shape
 
    idx_dtype = np.int32
 
    nnz_max = M*ntop
 
    indptr = np.zeros(M+1, dtype=idx_dtype)
    indices = np.zeros(nnz_max, dtype=idx_dtype)
    data = np.zeros(nnz_max, dtype=A.dtype)

    ct.sparse_dot_topn(
        M, N, np.asarray(A.indptr, dtype=idx_dtype),
        np.asarray(A.indices, dtype=idx_dtype),
        A.data,
        np.asarray(B.indptr, dtype=idx_dtype),
        np.asarray(B.indices, dtype=idx_dtype),
        B.data,
        ntop,
        lower_bound,
        indptr, indices, data)

    return csr_matrix((data,indices,indptr),shape=(M,N))

ERROR: Invalid requirement: '#uncomment'
You should consider upgrading via the 'e:\python\python37\python.exe -m pip install --upgrade pip' command.


ModuleNotFoundError: No module named 'sparse_dot_topn'

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

org_names = names['buyer'].unique()
vectorizer = TfidfVectorizer(min_df=1, analyzer=ngrams)
tf_idf_matrix = vectorizer.fit_transform(org_names)

In [ ]:
import time
t1 = time.time()
matches = awesome_cossim_top(tf_idf_matrix, tf_idf_matrix.transpose(), 10, 0.85)
t = time.time()-t1
print("SELFTIMED:", t)